In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np
import pathlib
import os
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


In [6]:
# -------------------------------
# 🔧 Config Flags
# -------------------------------
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
AUTOTUNE = tf.data.AUTOTUNE

N_COMPONENTS = 100
REPEAT_AUGMENTATION = 1

ENABLE_SCALING = True
ENABLE_PCA = True
ENABLE_SMOTE = True
ENABLE_CLASS_WEIGHT = True
ENABLE_AUGMENTATION = True

USE_CLASSIFIERS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'SVM': SVC(kernel='rbf', probability=True, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED)
}


In [7]:
ALL_RESULTS = {}

#ُSplit Data

In [8]:
import os
import pathlib
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [9]:
# train_ds, val_ds, test_ds, class_names = get_stratified_datasets(
#     root_dir='/content/drive/MyDrive/ROP-Data/10_split_dataset',
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     seed=42,
#     train_split=0.7,
#     val_split=0.1,
#     test_split=0.2
# )

In [10]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd
import pathlib


def paths_to_dataset(paths, labels, image_size, batch_size, training=False):
    """
    Convert file paths to TensorFlow dataset.
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        # Shuffle the dataset for training
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)

    def _load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)
        return img, label

    # Map the function, batch, cache, and prefetch
    ds = ds.map(_load_image, num_parallel_calls=AUTOTUNE).batch(batch_size).cache().prefetch(AUTOTUNE)

    return ds
def groupwise_stratified_split_from_dir(root_dir, train=0.7, val=0.1, test=0.2, seed=42):
    """
    Group-wise split of dataset (based on patient_id extracted from filename).
    Avoids data leakage between train/val/test. Uses GroupShuffleSplit.
    """
    assert abs(train + val + test - 1.0) < 1e-6, "Splits must sum to 1.0"

    root = pathlib.Path(root_dir)
    class_names = sorted([d.name for d in root.iterdir() if d.is_dir()])
    records = []

    for label_index, class_name in enumerate(class_names):
        for p in (root / class_name).rglob('*'):
            if p.suffix.lower() in EXTS:
                fname = p.name
                patient_id = fname.split('_')[0]  # extract patient ID
                records.append({
                    'path': str(p),
                    'label': label_index,
                    'patient_id': patient_id
                })

    df = pd.DataFrame(records)
    patient_ids = df['patient_id'].unique()
    n_groups = len(patient_ids)

    if n_groups < 3:
        raise ValueError(f"Too few patient groups ({n_groups}) to perform a 3-way split.")

    # ✅ Calculate group counts directly
    n_train = int(train * n_groups)
    n_val = int(val * n_groups)
    n_test = n_groups - n_train - n_val  # ensure total matches

    if n_train + n_val + n_test > n_groups:
        raise ValueError("Train + Val + Test groups exceed total number of groups.")

    # Shuffle patient IDs consistently
    rng = np.random.RandomState(seed)
    shuffled_patients = rng.permutation(patient_ids)

    train_patients = shuffled_patients[:n_train]
    val_patients = shuffled_patients[n_train:n_train + n_val]
    test_patients = shuffled_patients[n_train + n_val:]

    df_train = df[df['patient_id'].isin(train_patients)]
    df_val = df[df['patient_id'].isin(val_patients)]
    df_test = df[df['patient_id'].isin(test_patients)]

    X_train, y_train = df_train['path'].values, df_train['label'].values
    X_val, y_val = df_val['path'].values, df_val['label'].values
    X_test, y_test = df_test['path'].values, df_test['label'].values

    print("✅ Group-wise Stratified Split Summary:")
    print(" 📁 Train:", dict(zip(class_names, np.bincount(y_train))))
    print(" 📁 Val:  ", dict(zip(class_names, np.bincount(y_val))))
    print(" 📁 Test: ", dict(zip(class_names, np.bincount(y_test))))
    print(f" 👥 Patients → Train: {len(train_patients)}, Val: {len(val_patients)}, Test: {len(test_patients)}")

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), class_names


In [11]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), class_names = groupwise_stratified_split_from_dir(
    root_dir='/content/drive/MyDrive/ROP-Data/10_split_dataset',
    train=0.7,
    val=0.1,
    test=0.2,
    seed=42
)


✅ Group-wise Stratified Split Summary:
 📁 Train: {'Negative': np.int64(878), 'Positive': np.int64(403)}
 📁 Val:   {'Negative': np.int64(130), 'Positive': np.int64(39)}
 📁 Test:  {'Negative': np.int64(241), 'Positive': np.int64(121)}
 👥 Patients → Train: 131, Val: 18, Test: 39


In [12]:
train_ds = paths_to_dataset(X_train, y_train, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=True)
val_ds   = paths_to_dataset(X_val, y_val, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
test_ds  = paths_to_dataset(X_test, y_test, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)


# Piplines

In [13]:
# -------------------------------
# 📥 Feature Extraction Function
# -------------------------------
def extract_features(dataset, feature_extractor, repeat=1):
    features, labels = [], []
    for _ in range(repeat):
        for images, labs in dataset:
            feats = feature_extractor(images, training=False)
            features.append(feats.numpy())
            labels.append(labs.numpy())
    return np.concatenate(features), np.concatenate(labels)

In [14]:
def extract_features_pipeline(feature_extractor, name="ResNet50"):
    """
    Extract features and labels from datasets using the provided feature extractor.
    Returns variables directly (without saving to globals()).
    Also records and returns time for extraction.
    """
    print(f"\n📦 Extracting features using {name}...")

    start_time = time.time()

    X_train, y_train = extract_features(train_ds, feature_extractor, repeat=REPEAT_AUGMENTATION if ENABLE_AUGMENTATION else 1)
    X_val, y_val = extract_features(val_ds, feature_extractor)
    X_test, y_test = extract_features(test_ds, feature_extractor)

    end_time = time.time()
    elapsed = round(end_time - start_time, 4)

    print(f"✅ Done Extracting ({name}) in {elapsed} seconds")
    print(f"  X_train: {X_train.shape}")
    print(f"  y_train: {y_train.shape}")
    print(f"  X_test:  {X_test.shape}")

    return X_train, y_train, X_val, y_val, X_test, y_test, elapsed


In [15]:
def classification_pipeline(X_train, y_train, X_val, y_val, X_test, y_test, model_name="model", extract_time=None):
    """
    Train and evaluate classifiers on test set and store core metrics.
    Only saves f1-score, recall, precision, roc_auc, and timing for test set.
    """
    print(f"\n🚀 Classification pipeline for: {model_name}")

    if ENABLE_SCALING:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

    if ENABLE_PCA:
        pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
        X_train = pca.fit_transform(X_train)
        X_val = pca.transform(X_val)
        X_test = pca.transform(X_test)

    if ENABLE_SMOTE:
        smote = SMOTE(random_state=SEED)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    model_results = {}

    for name, model in USE_CLASSIFIERS.items():
        print(f"\n🔧 Training {name}...")

        if ENABLE_CLASS_WEIGHT and hasattr(model, 'class_weight'):
            if 'class_weight' in model.get_params():
                model.set_params(class_weight='balanced')

        if isinstance(model, XGBClassifier):
            neg, pos = np.bincount(y_train)
            model.set_params(scale_pos_weight=neg / pos if pos > 0 else 1)

        start_train = time.time()
        model.fit(X_train, y_train)
        end_train = time.time()

        start_test = time.time()
        y_pred = model.predict(X_test)
        end_test = time.time()

        # ROC-AUC (در صورت وجود)
        roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) if hasattr(model, "predict_proba") else None

        # فقط متریک‌های مورد نظر از report
        report = classification_report(y_test, y_pred, target_names=["Negative", "Positive"], digits=4, output_dict=True)

        model_results[name] = {
            'precision': round(report['weighted avg']['precision'], 4),
            'recall': round(report['weighted avg']['recall'], 4),
            'f1_score': round(report['weighted avg']['f1-score'], 4),
            'roc_auc': round(roc_auc, 4) if roc_auc is not None else None,
            'accuracy': round(report['accuracy'], 4),
            'timing': {
                'train_time': round(end_train - start_train, 4),
                'test_time': round(end_test - start_test, 4),
                'extract_time': extract_time
            }
        }

    ALL_RESULTS[model_name] = model_results
    print(f"✅ Stored core metrics for {model_name}")


# Feature Extractors

In [16]:
# 🧠 ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
input_tensor = layers.Input(shape=(*IMAGE_SIZE, 3))
x = layers.Lambda(preprocess_input)(input_tensor)
resnet_base = ResNet50(include_top=False, weights='imagenet', input_tensor=input_tensor, pooling='avg')
resnet_base.trainable = False
resnet_feature_extractor = models.Model(inputs=input_tensor, outputs=resnet_base.output)

# 🧠 EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
input_tensor = layers.Input(shape=(*IMAGE_SIZE, 3))
x = layers.Lambda(preprocess_input)(input_tensor)
efficientnet_base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_tensor, pooling='avg')
efficientnet_base.trainable = False
efficientnet_feature_extractor = models.Model(inputs=input_tensor, outputs=efficientnet_base.output)

# # 🧠 DenseNet121
# from tensorflow.keras.applications import DenseNet121
# from tensorflow.keras.applications.densenet import preprocess_input
# input_tensor = layers.Input(shape=(*IMAGE_SIZE, 3))
# x = layers.Lambda(preprocess_input)(input_tensor)
# densenet_base = DenseNet121(include_top=False, weights='imagenet', input_tensor=input_tensor, pooling='avg')
# densenet_base.trainable = False
# densenet_feature_extractor = models.Model(inputs=input_tensor, outputs=densenet_base.output)

# 🧠 VGG16
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
input_tensor = layers.Input(shape=(*IMAGE_SIZE, 3))
x = layers.Lambda(preprocess_input)(input_tensor)
vgg_base = VGG16(include_top=False, weights='imagenet', input_tensor=input_tensor)
vgg_base.trainable = False
x = vgg_base.output
x = layers.GlobalAveragePooling2D()(x)
vgg_feature_extractor = models.Model(inputs=input_tensor, outputs=x)

# 🧠 InceptionV3
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
input_tensor = layers.Input(shape=(*IMAGE_SIZE, 3))
x = layers.Lambda(preprocess_input)(input_tensor)
inception_base = InceptionV3(include_top=False, weights='imagenet', input_tensor=input_tensor, pooling='avg')
inception_base.trainable = False
inception_feature_extractor = models.Model(inputs=input_tensor, outputs=inception_base.output)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# 🟦 ResNet50


In [17]:
# 🟦 ResNet50
X_train_resnet, y_train_resnet, X_val_resnet, y_val_resnet, X_test_resnet, y_test_resnet, extract_time_resnet = extract_features_pipeline(resnet_feature_extractor, name="ResNet50")



📦 Extracting features using ResNet50...
✅ Done Extracting (ResNet50) in 504.9073 seconds
  X_train: (1281, 2048)
  y_train: (1281,)
  X_test:  (362, 2048)


In [18]:
classification_pipeline(
    X_train_resnet, y_train_resnet,
    X_val_resnet, y_val_resnet,
    X_test_resnet, y_test_resnet,
    model_name="ResNet50",
    extract_time=extract_time_resnet
)



🚀 Classification pipeline for: ResNet50

🔧 Training Logistic Regression...

🔧 Training SVM...

🔧 Training Random Forest...

🔧 Training XGBoost...

🔧 Training KNN...

🔧 Training Gradient Boosting...
✅ Stored core metrics for ResNet50


# 🟩 EfficientNetB0


In [19]:
# 🟩 EfficientNetB0
X_train_efficientnet, y_train_efficientnet, X_val_efficientnet, y_val_efficientnet, X_test_efficientnet, y_test_efficientnet, extract_time_efficientnet = extract_features_pipeline(efficientnet_feature_extractor, name="EfficientNetB0")



📦 Extracting features using EfficientNetB0...
✅ Done Extracting (EfficientNetB0) in 192.4802 seconds
  X_train: (1281, 1280)
  y_train: (1281,)
  X_test:  (362, 1280)


In [20]:
classification_pipeline(
    X_train_efficientnet, y_train_efficientnet,
    X_val_efficientnet, y_val_efficientnet,
    X_test_efficientnet, y_test_efficientnet,
    model_name="EfficientNetB0",
    extract_time=extract_time_efficientnet
)



🚀 Classification pipeline for: EfficientNetB0

🔧 Training Logistic Regression...

🔧 Training SVM...

🔧 Training Random Forest...

🔧 Training XGBoost...

🔧 Training KNN...

🔧 Training Gradient Boosting...
✅ Stored core metrics for EfficientNetB0


# 🟥 VGG16

In [ ]:
# 🟥 VGG16
X_train_vgg, y_train_vgg, X_val_vgg, y_val_vgg, X_test_vgg, y_test_vgg, extract_time_vgg = extract_features_pipeline(vgg_feature_extractor, name="VGG16")



📦 Extracting features using VGG16...
✅ Done Extracting (VGG16) in 1032.0511 seconds
  X_train: (1281, 512)
  y_train: (1281,)
  X_test:  (362, 512)


In [ ]:
classification_pipeline(
    X_train_vgg, y_train_vgg,
    X_val_vgg, y_val_vgg,
    X_test_vgg, y_test_vgg,
    model_name="VGG16",
    extract_time=extract_time_vgg
)



🚀 Classification pipeline for: VGG16

🔧 Training Logistic Regression...

🔧 Training SVM...

🔧 Training Random Forest...

🔧 Training XGBoost...

🔧 Training KNN...

🔧 Training Gradient Boosting...
✅ Stored core metrics for VGG16


# 🟪 InceptionV3

In [ ]:
# 🟪 InceptionV3
X_train_inception, y_train_inception, X_val_inception, y_val_inception, X_test_inception, y_test_inception, extract_time_inception = extract_features_pipeline(inception_feature_extractor, name="InceptionV3")



📦 Extracting features using InceptionV3...
✅ Done Extracting (InceptionV3) in 278.6608 seconds
  X_train: (1281, 2048)
  y_train: (1281,)
  X_test:  (362, 2048)


In [ ]:
classification_pipeline(
    X_train_inception, y_train_inception,
    X_val_inception, y_val_inception,
    X_test_inception, y_test_inception,
    model_name="InceptionV3",
    extract_time=extract_time_inception
)



🚀 Classification pipeline for: InceptionV3

🔧 Training Logistic Regression...

🔧 Training SVM...

🔧 Training Random Forest...

🔧 Training XGBoost...

🔧 Training KNN...

🔧 Training Gradient Boosting...
✅ Stored core metrics for InceptionV3


# Results

In [ ]:
print("📊 Final Summary of All Models:\n")

for model_name, results in ALL_RESULTS.items():
    print(f"\n🧠 Feature Extractor: {model_name}")
    for clf_name, data in results.items():
        print(f"  🔸 Classifier: {clf_name}")
        print(f"    - Accuracy     : {data.get('accuracy', 'N/A')}")
        print(f"    - Precision    : {data.get('precision', 'N/A')}")
        print(f"    - Recall       : {data.get('recall', 'N/A')}")
        print(f"    - F1-Score     : {data.get('f1_score', 'N/A')}")
        print(f"    - ROC-AUC      : {data.get('roc_auc', 'N/A')}")
        print(f"    - Extract Time : {data['timing'].get('extract_time', 'N/A')}s")
        print(f"    - Train Time   : {data['timing'].get('train_time', 'N/A')}s")
        print(f"    - Test Time    : {data['timing'].get('test_time', 'N/A')}s")

📊 Final Summary of All Models:


🧠 Feature Extractor: ResNet50
  🔸 Classifier: Logistic Regression
    - Accuracy     : 0.7652
    - Precision    : 0.8215
    - Recall       : 0.7652
    - F1-Score     : 0.7718
    - ROC-AUC      : 0.9119
    - Extract Time : 504.9073s
    - Train Time   : 0.0767s
    - Test Time    : 0.0006s
  🔸 Classifier: SVM
    - Accuracy     : 0.8011
    - Precision    : 0.8266
    - Recall       : 0.8011
    - F1-Score     : 0.806
    - ROC-AUC      : 0.9016
    - Extract Time : 504.9073s
    - Train Time   : 0.9081s
    - Test Time    : 0.0379s
  🔸 Classifier: Random Forest
    - Accuracy     : 0.7983
    - Precision    : 0.7988
    - Recall       : 0.7983
    - F1-Score     : 0.7985
    - ROC-AUC      : 0.8828
    - Extract Time : 504.9073s
    - Train Time   : 1.9731s
    - Test Time    : 0.0121s
  🔸 Classifier: XGBoost
    - Accuracy     : 0.7901
    - Precision    : 0.8034
    - Recall       : 0.7901
    - F1-Score     : 0.7938
    - ROC-AUC      : 0.8693
 